In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install mlflow
!pip install tabpfn
import os
import random
from getpass import getpass

import mlflow
import mlflow.keras
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tabpfn import TabPFNClassifier
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

dataset_path = "/content/drive/MyDrive/Synthetic Banking Transaction Dataset with Multi-P/Synthetic Banking Transaction Dataset with Multi-P/Fraud Detection Dataset-bmJaU4/Fraud Detection Dataset.csv"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 22.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 47.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 43.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.0 MB/s  0:00:00
  Attempting uninstall: importlib_metadata
    Found existing installation: importlib_metadata 10.1.0
    Uninstalling importlib_metadata-10.1.0:
      Successfully uninstalled importlib_metadata-10.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18/18 [mlflow]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 850.0/850.0 kB 17.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 19.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [tabpfn]


In [4]:
df = pd.read_csv(dataset_path)

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())
print("Fraud types:")
print(df["fraud_type"].value_counts())
display(df.head())

Dataset shape: (1000000, 17)
Columns:
['transaction_id', 'timestamp', 'customer_id', 'card_id', 'device_id', 'ip_address', 'merchant_id', 'merchant_category', 'merchant_country', 'merchant_city', 'merchant_latitude', 'merchant_longitude', 'transaction_type', 'amount', 'currency', 'is_fraud', 'fraud_type']
Fraud types:
fraud_type
card_testing             800
geo_anomaly              400
account_takeover         210
money_laundering_ring     80
Name: count, dtype: int64


,transaction_id,timestamp,customer_id,card_id,device_id,ip_address,merchant_id,merchant_category,merchant_country,merchant_city,merchant_latitude,merchant_longitude,transaction_type,amount,currency,is_fraud,fraud_type
0,TXN000000000001,2025-10-19T07:46:29+00:00,CUST0018039,CARD0018039-2,ef4bb478f96c,165.185.21.33,MID003362,restaurants,CA,Taylorberg,-58.474226,116.522544,purchase,18.79,USD,0,NaN
1,TXN000000000002,2025-06-29T01:09:39+00:00,CUST0040966,CARD0040966-2,d57986988353,147.24.150.17,MID014052,electronics,GB,Brownhaven,-21.149589,-4.221602,purchase,19.98,USD,0,NaN
2,TXN000000000003,2025-07-06T10:28:29+00:00,CUST0055324,CARD0055324-2,5cb8294a9726,211.23.48.192,MID018641,fashion,BR,Butlerhaven,-51.118131,66.036870,purchase,21.25,USD,0,NaN
3,TXN000000000004,2025-01-26T01:49:36+00:00,CUST0023733,CARD0023733-3,3449ec767cf7,136.82.124.111,MID004962,fashion,DE,Sullivanborough,12.523283,-34.068509,purchase,61.70,USD,0,NaN
4,TXN000000000005,2025-06-04T11:27:59+00:00,CUST0039805,CARD0039805-1,56e9c34ba51c,82.46.29.202,MID009699,pharmacy,DE,Staceyfort,3.363100,178.930994,purchase,70.06,USD,0,NaN


In [5]:
df = df.drop(columns=["transaction_id", "customer_id", "card_id", "device_id", "merchant_id"])

ts = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
df["hour"] = ts.dt.hour
df["day"] = ts.dt.day
df["month"] = ts.dt.month
df["year"] = ts.dt.year

df = df.drop(columns=["timestamp", "ip_address", "merchant_city"])
display(df.head())

,merchant_category,merchant_country,merchant_latitude,merchant_longitude,transaction_type,amount,currency,is_fraud,fraud_type,hour,day,month,year
0,restaurants,CA,-58.474226,116.522544,purchase,18.79,USD,0,NaN,7,19,10,2025
1,electronics,GB,-21.149589,-4.221602,purchase,19.98,USD,0,NaN,1,29,6,2025
2,fashion,BR,-51.118131,66.036870,purchase,21.25,USD,0,NaN,10,6,7,2025
3,fashion,DE,12.523283,-34.068509,purchase,61.70,USD,0,NaN,1,26,1,2025
4,pharmacy,DE,3.363100,178.930994,purchase,70.06,USD,0,NaN,11,4,6,2025


In [6]:
categorical_cols = [
    "merchant_country",
    "merchant_category",
    "currency",
    "transaction_type",
]

numerical_cols = [
    "merchant_latitude",
    "merchant_longitude",
    "amount",
    "hour",
    "day",
    "month",
    "year",
]

In [7]:
normal_transactions = df[df["is_fraud"] == 0].loc[:50000].copy()
card_testing = df[
    (df["is_fraud"] == 1) & (df["fraud_type"] == "card_testing")
].copy()

print("Normal transactions:", normal_transactions.shape)
print("Held-out card-testing transactions:", card_testing.shape)

Normal transactions: (49917, 13)
Held-out card-testing transactions: (800, 13)


In [8]:
X = normal_transactions.drop(columns=["is_fraud", "fraud_type"])

X_train, temp = train_test_split(X, test_size=0.3, random_state=SEED)
X_val, X_test_normal = train_test_split(temp, test_size=0.5, random_state=SEED)

X_test = pd.concat(
    [X_test_normal, card_testing.drop(columns=["is_fraud", "fraud_type"])],
    axis=0,
)
Y_test = pd.Series(
    [0] * len(X_test_normal) + [1] * len(card_testing),
    index=X_test.index,
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)
print("Normal test transactions:", int((Y_test == 0).sum()))
print("Fraud test transactions:", int((Y_test == 1).sum()))

Training set: (34941, 11)
Validation set: (7488, 11)
Test set: (8288, 11)
Normal test transactions: 7488
Fraud test transactions: 800


In [9]:
scaler = StandardScaler()
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype=np.float32,
    min_frequency=20,
)

X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_val[numerical_cols] = scaler.transform(X_val[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

train_encoded = encoder.fit_transform(X_train[categorical_cols])
val_encoded = encoder.transform(X_val[categorical_cols])
test_encoded = encoder.transform(X_test[categorical_cols])

encoded_columns = encoder.get_feature_names_out(categorical_cols)

X_train = pd.concat(
    [
        X_train.drop(columns=categorical_cols).reset_index(drop=True),
        pd.DataFrame(train_encoded, columns=encoded_columns),
    ],
    axis=1,
)
X_val = pd.concat(
    [
        X_val.drop(columns=categorical_cols).reset_index(drop=True),
        pd.DataFrame(val_encoded, columns=encoded_columns),
    ],
    axis=1,
)
X_test = pd.concat(
    [
        X_test.drop(columns=categorical_cols).reset_index(drop=True),
        pd.DataFrame(test_encoded, columns=encoded_columns),
    ],
    axis=1,
)

In [10]:
mlflow.set_experiment("Isolation Forest")

with mlflow.start_run():
    Isolation_forest_model = IsolationForest(
        n_estimators=300,
        max_samples="auto",
        contamination="auto",
        random_state=SEED,
        verbose=0,
    )
    Isolation_forest_model.fit(X_train)

    val_scores = Isolation_forest_model.decision_function(X_val)
    threshold = float(np.quantile(val_scores, 0.1))

    test_scores = Isolation_forest_model.decision_function(X_test)
    y_pred = (test_scores < threshold).astype(int)

    precision = precision_score(Y_test, y_pred, zero_division=0)
    recall = recall_score(Y_test, y_pred, zero_division=0)
    f1 = f1_score(Y_test, y_pred, zero_division=0)
    accuracy = accuracy_score(Y_test, y_pred)
    roc_auc = roc_auc_score(Y_test, -test_scores)

    mlflow.log_params(
        {
            "n_estimators": 300,
            "max_samples": "auto",
            "contamination": "auto",
            "random_state": SEED,
            "training_data": "normal_only",
            "threshold_calibration": "normal_validation_10th_percentile",
        }
    )
    mlflow.log_metrics(
        {
            "anomaly_threshold": threshold,
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "accuracy": float(accuracy),
            "roc_auc": float(roc_auc),
        }
    )
    mlflow.sklearn.log_model(
        sk_model=Isolation_forest_model,
        name="isolation_forest",
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )

    print(f"Threshold:          {threshold:.6f}")
    print(f"Precision:          {precision:.4f}")
    print(f"Recall:             {recall:.4f}")
    print(f"F1 score:           {f1:.4f}")
    print(f"Accuracy:           {accuracy:.4f}")
    print(f"ROC-AUC:            {roc_auc:.4f}")
    print("Classification Report:", classification_report(Y_test, y_pred))

2026/10/29 13:50:42 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/29 13:50:42 INFO mlflow.store.db.utils: Updating database tables
2026/10/29 13:50:49 INFO mlflow.tracking.fluent: Experiment with name 'Isolation Forest' does not exist. Creating a new experiment.


Threshold:          0.022687
Precision:          0.1044
Recall:             0.1087
F1 score:           0.1066
Accuracy:           0.8240
ROC-AUC:            0.5285
Classification Report:               precision    recall  f1-score   support

           0       0.90      0.90      0.90      7488
           1       0.10      0.11      0.11       800

    accuracy                           0.82      8288
   macro avg       0.50      0.50      0.50      8288
weighted avg       0.83      0.82      0.83      8288



In [11]:
mlflow.set_experiment("Autoencoder")

with mlflow.start_run():
    n_features = X_train.shape[1]

    Autoencoder_model = keras.Sequential(
        [
            layers.Input(shape=(n_features,)),
            layers.Dense(64, activation="relu"),
            layers.Dense(32, activation="relu"),
            layers.Dense(16, activation="relu"),
            layers.Dense(32, activation="relu"),
            layers.Dense(64, activation="relu"),
            layers.Dense(n_features, activation="linear"),
        ]
    )
    Autoencoder_model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"],
    )
    Autoencoder_model.summary()

    history = Autoencoder_model.fit(
        X_train,
        X_train,
        validation_data=(X_val, X_val),
        epochs=30,
        batch_size=256,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True,
            )
        ],
    )

    train_loss = history.history["loss"][-1]
    val_loss = history.history["val_loss"][-1]

    val_rec = Autoencoder_model.predict(X_val, verbose=0)
    val_scores = np.mean(np.square(X_val - val_rec), axis=1)
    threshold = float(np.percentile(val_scores, 92))

    test_rec = Autoencoder_model.predict(X_test, verbose=0)
    test_scores = np.mean(np.square(X_test - test_rec), axis=1)
    y_pred = (test_scores > threshold).astype(int)

    precision = precision_score(Y_test, y_pred, zero_division=0)
    recall = recall_score(Y_test, y_pred, zero_division=0)
    f1 = f1_score(Y_test, y_pred, zero_division=0)
    accuracy = accuracy_score(Y_test, y_pred)
    roc_auc = roc_auc_score(Y_test, test_scores)
    avg_precision = average_precision_score(Y_test, test_scores)

    tn, fp, fn, tp = confusion_matrix(Y_test, y_pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) else 0.0

    mlflow.log_params(
        {
            "architecture": "64-32-16-32-64",
            "optimizer": "adam",
            "loss": "mse",
            "epochs": 30,
            "batch_size": 256,
            "early_stopping_patience": 5,
            "random_state": SEED,
            "training_data": "normal_only",
            "threshold_calibration": "normal_validation_92nd_percentile",
        }
    )
    mlflow.log_metrics(
        {
            "final_train_loss": float(train_loss),
            "final_val_loss": float(val_loss),
            "anomaly_threshold": threshold,
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "accuracy": float(accuracy),
            "roc_auc": float(roc_auc),
            "average_precision": float(avg_precision),
            "false_positive_rate": float(fpr),
            "true_positives": int(tp),
            "false_positives": int(fp),
            "true_negatives": int(tn),
            "false_negatives": int(fn),
        }
    )
    mlflow.keras.log_model(Autoencoder_model, name="autoencoder")

    print(f"Training loss:      {train_loss:.6f}")
    print(f"Validation loss:    {val_loss:.6f}")
    print(f"Threshold:          {threshold:.6f}")
    print(f"Precision:          {precision:.4f}")
    print(f"Recall:             {recall:.4f}")
    print(f"F1 score:           {f1:.4f}")
    print(f"Accuracy:           {accuracy:.4f}")
    print(f"ROC-AUC:            {roc_auc:.4f}")
    print(f"Average precision:  {avg_precision:.4f}")
    print("Classification Report:", classification_report(Y_test, y_pred, zero_division=0))

2026/10/29 13:51:01 INFO mlflow.tracking.fluent: Experiment with name 'Autoencoder' does not exist. Creating a new experiment.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 29)             │         1,885 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,069 (35.43 KB)

 Trainable params: 9,069 (35.43 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.1933 - mae: 0.2671 - val_loss: 0.0792 - val_mae: 0.1723
Epoch 2/30
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0643 - mae: 0.1552 - val_loss: 0.0558 - val_mae: 0.1443
Epoch 3/30
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0518 - mae: 0.1384 - val_loss: 0.0493 - val_mae: 0.1345
Epoch 4/30
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0478 - mae: 0.1339 - val_loss: 0.0461 - val_mae: 0.1323
Epoch 5/30
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0447 - mae: 0.1307 - val_loss: 0.0426 - val_mae: 0.1274
Epoch 6/30
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0414 - mae: 0.1259 - val_loss: 0.0411 - val_mae: 0.1252
Epoch 7/30
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0405 - mae: 0.1237 - val_loss: 0.0403 - val_mae: 0.1212
Epoch 8/30
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0397 - mae: 0.1222 - val_loss: 0.0395 - val_mae: 0.1227
Epoch 9/30
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - lo

2026/10/29 13:51:16 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


Training loss:      0.025887
Validation loss:    0.025844
Threshold:          0.040349
Precision:          0.1028
Recall:             0.0813
F1 score:           0.0908
Accuracy:           0.8429
ROC-AUC:            0.4884
Average precision:  0.0955
Classification Report:               precision    recall  f1-score   support

           0       0.90      0.92      0.91      7488
           1       0.10      0.08      0.09       800

    accuracy                           0.84      8288
   macro avg       0.50      0.50      0.50      8288
weighted avg       0.83      0.84      0.83      8288



In [12]:
X_tabpfn_context = pd.concat(
    [card_testing.iloc[:5], normal_transactions.iloc[:5]],
)
y_tabpfn_context = pd.Series(
    [1] * 5 + [0] * 5,
    index=X_tabpfn_context.index,
)

tabpfn_test = pd.concat(
    [card_testing.iloc[5:790], normal_transactions.iloc[5:790]],
)
X_tabpfn_test = tabpfn_test.drop(columns=["is_fraud", "fraud_type"])
y_tabpfn_test = tabpfn_test["is_fraud"]
X_tabpfn_context = X_tabpfn_context.drop(columns=["is_fraud", "fraud_type"])

print("TabPFN context size:", X_tabpfn_context.shape)
print("TabPFN test size:", X_tabpfn_test.shape)
print("Fraud test transactions:", int((y_tabpfn_test == 1).sum()))
print("Normal test transactions:", int((y_tabpfn_test == 0).sum()))

TabPFN context size: (10, 11)
TabPFN test size: (1570, 11)
Fraud test transactions: 785
Normal test transactions: 785


In [13]:
os.environ["TABPFN_TOKEN"] = getpass("Enter TabPFN token: ")

Enter TabPFN token: ··········


In [14]:
tabpfn_model = TabPFNClassifier()
tabpfn_model.fit(X_tabpfn_context, y_tabpfn_context)

tabpfn_pred = tabpfn_model.predict(X_tabpfn_test)
print(classification_report(y_tabpfn_test, tabpfn_pred, zero_division=0))

tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

              precision    recall  f1-score   support

           0       1.00      0.74      0.85       785
           1       0.79      1.00      0.88       785

    accuracy                           0.87      1570
   macro avg       0.90      0.87      0.87      1570
weighted avg       0.90      0.87      0.87      1570

